# row-check — distributed Stage 1

In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
CONFIG = ROOT / 'examples/e2e/config.json'
RUN_S3 = os.getenv('DTRACK_RUN_S3', 's3://dtrack-e2e-work/dtrack/v1/runs')
RUN_ID = os.getenv('DTRACK_RUN_ID', 'nb-distributed')
PAIR = 'orders'
EXECUTE = False  # True needs the configured Left credentials and RUN_S3 access.

subprocess.run([sys.executable, str(ROOT / 'examples/e2e/make_synthetic.py')], cwd=ROOT, check=True, capture_output=True)
case = json.loads(CONFIG.read_text())
def source_name(side):
    return side.get('source') or case['connections'][side['conn_macro']]['source']

{name: {'left': source_name(p['left']), 'right': source_name(p['right']), 'sample': p.get('sample')} for name, p in case['pairs'].items()}

In [ ]:
def dtrack(*args, execute=EXECUTE):
    command = [sys.executable, '-m', 'dtrack', *map(str, args)]
    print('$', shlex.join(command))
    return subprocess.run(command, cwd=ROOT, text=True, check=execute) if execute else command

# Local/operator producer: counts Left partitions and writes row_check/*.left.json.
row_command = dtrack('row-check', '--config', CONFIG, '--run-s3', RUN_S3, '--run-id', RUN_ID, '--pair', PAIR)
row_command

In [ ]:
# Databricks consumer: start src/dtrack_right/notebook.py as one Job before or after row-check.
# Asset Bundles may deploy/run that notebook; SQL Warehouse is not the distributed consumer.
workspace_target = os.getenv('DTRACK_DBX_WORKSPACE_DIR', '/Shared/dtrack/src/dtrack_right')
deployment = {
    'workspace_upload': f'dtrack tools dbx workspace upload src/dtrack_right {workspace_target} --overwrite',
    'job_notebook': f'{workspace_target}/notebook',
    'job_parameters': {'run_s3': RUN_S3, 'run_id': RUN_ID, 'gate': 'auto'},
    'asset_bundle': 'databricks bundle deploy -t dev && databricks bundle run dtrack_right -t dev',
}
deployment

In [ ]:
# Expected handoff: Left documents -> Right counts/gate -> manifest/<pair>.json + row_check/_CLOSED.
dtrack('tools', 'fetch-report', '--help', execute=True) if False else {'next': '01_col_check.ipynb', 'run_id': RUN_ID}

In [ ]:
# Unit contract on the real case: source counts, reconciliation, gate, and known injected holes.
from dtrack_left.backends.sources import FileSource
from dtrack_right.checks.row_check import evaluate_gate, reconcile_partitions, summarize_partitions

left_cfg = {**case['pairs'][PAIR]['left'], 'path': str(ROOT / case['pairs'][PAIR]['left']['path'])}
right_cfg = {'source': 'file', 'path': str(ROOT / 'examples/e2e/data/right/orders.csv'), 'date_col': 'business_date', 'date_type': 'string'}
left_counts = FileSource(left_cfg).partition_counts('2026-01-01', '2026-02-28')
right_counts = FileSource(right_cfg).partition_counts('2026-01-01', '2026-02-28')
partitions = reconcile_partitions(left_counts, right_counts)
assert summarize_partitions(partitions) == {'match': 56, 'count_diff': 1, 'left_only': 1, 'right_only': 1}
assert partitions['2026-01-15']['delta'] == -3
assert partitions['2026-01-31']['verdict'] == 'right_only'
assert partitions['2026-02-10']['verdict'] == 'left_only'
assert evaluate_gate(partitions, 'auto')['passed'] is True
summarize_partitions(partitions)